https://habrahabr.ru/company/ods/blog/327250/

# ??????????? ?????????? ???????????? ????????

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
sns.set()
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
def get_preprocessed_titanic():
    df = sns.load_dataset('titanic')

    X, y = df.drop(['survived'], axis=1), df.survived

    X["embarked"] = X["embarked"].fillna("S")
    X["fare"].fillna(X["fare"].median(), inplace=True)
    X["embark_town"] = X["embark_town"].fillna(X.embark_town.value_counts().index[0]) 
    
    average_age_titanic   = X["age"].mean()
    std_age_titanic       = X["age"].std()
    count_nan_age_titanic = X["age"].isnull().sum()
    rand_1 = np.random.randint(average_age_titanic - std_age_titanic, 
                               average_age_titanic + std_age_titanic, 
                               size=count_nan_age_titanic)
    X["age"][np.isnan(X["age"])] = rand_1

    X.drop(['deck', 'alive'], axis=1, inplace=True)
    
    return X, y

In [ ]:
X, y = get_preprocessed_titanic()

In [ ]:
X = pd.get_dummies(X)

In [ ]:
from sklearn.model_selection import GridSearchCV, cross_val_score

In [ ]:
SEED = 192837465
np.random.seed = SEED

## Sklearn

### AdaBoost

In [ ]:
from sklearn.ensemble import AdaBoostClassifier

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

In [ ]:
scores = cross_val_score(LogisticRegression(), X.values, y)
np.mean(scores), 2*np.std(scores)

In [ ]:
scores = cross_val_score(AdaBoostClassifier(LogisticRegression(), random_state=SEED), X, y)
np.mean(scores), 2*np.std(scores)

In [ ]:
scores = cross_val_score(AdaBoostClassifier(DecisionTreeClassifier(random_state=SEED), random_state=SEED), X, y)
np.mean(scores), 2*np.std(scores)

In [ ]:
scores = cross_val_score(AdaBoostClassifier(RandomForestClassifier(random_state=SEED), random_state=SEED), X, y)
np.mean(scores), 2*np.std(scores)

##### ????? ????????? ??? ????????? ?????

* __`n_estimators`__ - ??????? ?? ???? ???????? ???????? ?????????, ?????? ????? ????????????? ?? ???????? ???????.
* __`learning_rate`__ - ???????? ????????

##### ??? ????????????

* ??????????? ??????? ????????????? ??? ????????.
* ???? ??????? ????????????? ?? ?????? ???? ????? ????? ? ?????????? ???????????, ?? ????? ??? ?????? ??????????????.

### GBM

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

In [ ]:
clf = GradientBoostingClassifier()

In [ ]:
scores = cross_val_score(GradientBoostingClassifier(random_state=SEED), X, y)
np.mean(scores), 2*np.std(scores)

##### ????? ????????? ??? ??????????

??????????? ? ????????:

* __`min_samples_split`__ - ?&#243;????? ????????, ???????, ???????? ?? ?????? ??????? ?? ??????? ????????, ?? ??????? ??????? ???????? ????? ???????? ? ????????????. ????? ????????? ?? ???????? ~0.5-1% ?? ????? ???????? ? ???????. 
    * _???????????? CV ??? ?????????._


* __`min_samples_leaf`__ - ??? ? `min_samples_split`, ???????? ??????????????? ????????????. ? ????? _??????? ???????? ????? ????????????, ????? ???? ???? ? ?????????????????? ???????_, ??? ??? ? ???????, ?????????? ??? ?????? ??????, ?????????.
    * _???????????? CV ??? ?????????._
    

* __`min_weight_fraction_leaf`__ - ????? ?? `min_samples_leaf`, ?? ????????????? ??? _????_ ?? ?????? ????? ???????? ?????? _?????_. ???????????? ??? ???? ????????, ??? `min_samples_leaf` ?? ??????????.
    * _???????????? CV ??? ?????????._


* __`max_depth`__ - ???????????? ??????? ??????. ???????????? ??? ???????? ????????????, ??? ??? ?&#243;????? ??????? ????????? ?????? ??????? ??????????? ??????????? ???????????? ??????? ?? ???????? ???????.
    * ?????? ? ????????? 3-10.
    * _???????????? CV ??? ?????????._


* __`max_leaf_nodes`__ - ???????????? ????? ???????????? ?????, ?? ???? ?????? ??????. ????? ???? ???????????? ?????? `max_depth`. ?.?. ??????? ????????, ?????? ???????? $N$ ????????? _?? ?????_ $2^N$ ??????. ???? ????? ???? ????????, GBM ????? ???????????? `max_depth`.
    * _???????????? CV ??? ?????????._


* __`max_features`__ - ?????????? ?????????, ??????? ??????????????? ??? ?????? ??????? ??????. ???????? ??? ???? ?????????? ????????. ??? ???????, $\sqrt{M}$, ??? $M$ - ????? ?????????, ???????? ???????, ?? ????? ??????????? ??????? ??? ??????? ?? 30-40% ?? ?????? ????? ?????????. 
    * `NB`: ??????? ???????? ????? ???????? ? ????????????.
    
??????????? ? ????????:

* __`learning_rate`__ - ?????? ???????? ?????????? ????? ??????? ?? ???????? ?? ????????? ????????????. 
    * ???????? ???????? ???????? ??????? ?????? ?????? ??????? ? ????????? ?????????.
    * ??????? ???????? ????? ???????????????, ??? ??? ??? ?????? ?????? ????? ????????? ? ????????????? ?????????? ?????? ? ????? ??????? ???????? ??? ?????????? ???????????.
    * ? ?? ?? ????? ??????? ???????? ????? ?? ????? ??????? ?????????? ????????, ??? ?????? ????? ????????? ?????????? ??????.


* __`n_estimators`__ - ?????????? ????????, ??????? ????? ?????????. 
    * ???????? ?? ??, ??? GBM ?????????? ???????? ??? ??????? ?????????? ????????, ?? ??? ????? ????? ?????????????. 
    * ???????????? ???????????? `learning_rate` ? ??????? ??????????? ?????????? ???????? ? ??????? CV.


* __`subsample`__ - ???? ????????, ??????? ?????????? ??? ???????? ??????? ?? ???????? (?????????? ??????????? ??????? ????????? ???????).
    * ???????? ???? ??????? 1 ?????? ?????? ?????????, ??? ??? ????????? ?????????????? ??????????.
    * ?????? ???????? ~0.8 ???????? ????.

##### ??? ????????????

1. ???????? ???????????? ??????? ???????? ???????? (`learning rate`). ?????? ??? 0.1, ?? ? ?????? ???? ????? ??????? ?? ????????? 0.05-0.2.

2. ???????? ??????????? ?????????? ???????? ??? ??????????????? ???????? ????????. ?????????????? ?? 40-70. ? ??? ????? ?? ???? ???? ???? ????? ????? ????? ??????????, ??????? ??? ?????????? ????????? ?????? ?? ?????????? ??????????, ??? ??? ??? ???????????? ??? ??? ???????????? ??? ???????? ????????? ????????? (?.3).

3. ????????? ??????????? ????????? ????????, ???????????? `learning_rate` ? `n_estimators`.

4. ????????? `learning_rate`, ????????? `n_estimators`, ????? ???????? ????? ????????? ??????.

## XGBoost

In [ ]:
from xgboost import XGBClassifier, XGBRegressor

In [ ]:
clf = XGBClassifier()

In [ ]:
scores = cross_val_score(XGBClassifier(seed=SEED), X, y)
np.mean(scores), 2*np.std(scores)

??? XGBoost ??????????, ??? GBM?

1. ????????????? ?? ???????.
2. ???????????? ?? ???? ? ?????????? ?? "???".
3. ????? ???????? ? ???????????? ? ?????????? ? ?????????.
4. ????????????? ???????? ?????-????????? ????? ?????? ?? ????????, ??? ????????? ?? ???????????? ?????? ??????, ??????? ???????? ???????? ??????????.

##### ????? ????????? ??? ??????????

* __`booster [default=gbtree]`__ - ??? ??????. ????? 2 ????????: 
    * `gbtree` - ???????
    * `gblinear` - ???????? ??????

?????? ???? ?????? ?????? ? ???????? ????????, ??? ??? ??? ????? ?????? ($\lim\rightarrow1)$ ??????????? ???????? ??????.

??????????? ? ????????:

* __`eta [default=0.3]`__ - ?????? `learning_rate`. ???? ?? ??????????? sklearn-???????? ??????? ????? XGBClassifier, ?? ???? ???????? ? ????? ?????????? `learning_rate`.
    * ??????? ?????????? ?? ????????? 0.01-0.2.


* __`min_child_weight [default=1]`__ - ?????????? ??????????? ????? ????? ???? ???????? ? ???????? ????. ???????? ????? ?? `min_child_leaf` ? GBM, ?? ?? ??????: ??????? ? ???? ????? ????? ????????, ? ?? ????? ??? ? GBM ??????? ? ???? ?????? ?????????? ????????.
    * ????????? ????????? ?????????????: ?&#243;????? ???????? ?? ???? ?????? ??????????? ??????? ??????? ??????????? ????? ?????????.
    * ??????? ?????? ???????? ????? ? ????????????.
    * _???????????? CV ??? ?????????._
    

* __`max_depth [default=6]`__ - ???????, ????? ??, ??? ? ? GBM.
    * _???????????? CV ??? ?????????._
    * ???????? ????????: 3-10.
    

* __`max_leaf_nodes`__ - ?? ??, ??? ? ? GBM. ??. GBM.


* __`gamma [default=0]`__ - ???? ?????? ????? ????????, ???? ?????? ? ?????????? ????? ?????????? ????? ????????? ???????? ??????? ??????. ???????? $\gamma$ ?????? ??????? ???????? ????? ??????????. ????? ???????? ?????? ???????? ????? ??????????????. ????????, ????????, ????????? ?? ????????? ? ?????? ?????? ??????? ??????, ? ?????? ???? ??????? ?????.


* __`max_delta_step [default=0]`__ - ?????????? ??? (??????, ?????) ??????? ?????? ???? ?? ?????? ?????????? ??????. ? ?????????? ??????????? ?????????? ? ???????? ?????????? ?????????, ? ?????????? ????? ???????. ???-??-?????, ???? ???????? ?????? ????????? ?????? ???????? ?? ?????????, ???? ? ??????? _????? ?????????????????? ???????_ ????? ??????.


* __`subsample [default=1]`__ - ?? ??, ??? ? ? GBM. 
    * ?????? 0.5-1.


* __`colsample_bytree [default=1]`__ - ?? ??, ??? ? `max_features` ? GBM.
    * ? XGB ??????? ? ????????? 0.5-1.


* __`colsample_bylevel [default=1]`__ - ?????? ?? ????????????, ??? ?????? ????? ?? ???? ?????????? ????????.


* __`lambda [default=1]`__ - ?? ??, ??? ? $\alpha$ ? $L_2$-????????????? ????????? Ridge. ?????? ?? ????????????. ???? ?? ??????????? sklearn-???????? ??????? ????? XGBClassifier, ?? ???? ???????? ? ????? ?????????? `reg_lambda`.


* __`alpha [default=0]`__ - ?? ??, ??? ? $\alpha$ ? $L_1$-????????????? ????????? Lasso. ????? ???? ??????????? ? ????? ??????????????? ??????????? ????????????. ???? ?? ??????????? sklearn-???????? ??????? ????? XGBClassifier, ?? ???? ???????? ? ????? ?????????? `reg_alpha`.


* __`scale_pos_weight [default=1]`__ - ????????, ??????? ??? 0, ????? ???? ???????????? ? ?????? ????? ?????????????????? ??????? ??? ????? ???????? ????????.

##### ??? ????????????

1. ???????? ???????????? ??????? ???????? ???????? (`learning rate`). ?????? ??? 0.1, ?? ? ?????? ???? ????? ??????? ?? ????????? 0.05-0.3.

2. ???????? ??????????? ?????????? ???????? ??? ??????????????? ???????? ????????. ? XGB ???? ???????? ??????? `cv`, ??????? ???????? ???????????? ???????? ????? ?????? ?? ???????? ???????? ? ?????????? ??????????? ?????????? ????????.

3. ????????? ??????????? ????????? ????????, ???????????? `learning_rate` ? `n_estimators`:
    * max_depth
    * min_child_weight 
    * gamma 
    * subsample 
    * colsample_bytree

4. ????????? ????????? ?????????????:
    * reg_lambda
    * reg_alpha

5. ????????? `learning_rate`, ???????? ?????? ??????.

# LightGBM

In [ ]:
from lightgbm import LGBMClassifier

In [ ]:
scores = cross_val_score(LGBMClassifier(seed=SEED), X, y)
np.mean(scores), 2*np.std(scores)

??? ????????? LightGBM?

* ??????? ??????????? ??????
* ???????, ??? ?? ???????? ?? ????, ? ?? ? ?????
* ????? ??????????? ?? GPU
* ? ??????? ????? ??????? ?????? ??????

LightGBM ?????? ?????? ?? ?????????? ?????? ?? ???????, ? ?? ??????? ?? ???????, ?? ???? ???? ?????????? ? ????????????. ??????? ?????? ???????? ? ????????? ??????? ?????? ?? __`num_leaves`__.

????????? ? ??????????? ????? ????????????? ?? ???????? ? XGB ? GBM. ?????? ????? ???? ????????? ????????????:

##### ????? ??????? ??????? ????????

1. ??????????? ??????? ???????? __`max_bin`__.
2. ?????????? __`learning_rate`__, ???????????? __`num_iterations`__.
3. ????? ??????????? ????????? __`num_leaves`__, ?? ????? ???????? ?? ?????????????.

##### ????? ???????? ? ?????????????

* ?????????? __`max_bin`__
* ?????????? __`num_leaves`__
* ??????????? __`min_data_in_leaf`__ ? __`min_sum_hessian_in_leaf`__
* ??????????? ???????, ????? ????????? __`bagging_fraction`__ ? __`bagging_freq`__
* ??????????? ??????????????? ????????? ??????? ??????????? ????? __`feature_fraction`__
* ????????????? ????? ?????????????? ? ??????? ?????????? __`lambda_l1`__, __`lambda_l2`__ ? __`min_gain_to_split`__
* ???????????? __`max_depth`__

# H2O

??????? ??????, ??? ??? ??? ??????? LightGBM ? ?????????????? ???????????.

http://docs.h2o.ai/h2o/latest-stable/h2o-docs/data-science/gbm.html#defining-a-gbm-model

# CatBoost

https://habrahabr.ru/company/yandex/blog/333522/

In [ ]:
from catboost import Pool, CatBoostClassifier

In [ ]:
scores = cross_val_score(CatBoostClassifier(random_seed=SEED), X.values, y)
np.mean(scores), 2*np.std(scores)

In [ ]:
clf = CatBoostClassifier()

In [ ]:
# clf.fit(X.values, y, plot=True)